# Шпора за 1 день

Самый минимум. Каждая ячейка -- готовый копипаст.

---
## 0. Старт любого ноутбука

In [ ]:
import random, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, classification_report, mean_squared_error

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
TARGET = 'target'  # <--
ID_COL = 'id'      # <--

---
## 1. Таблица -> CatBoost (быстрый бейзлайн)

In [ ]:
from catboost import CatBoostClassifier  # или CatBoostRegressor

features = [c for c in train.columns if c not in [TARGET, ID_COL]]
X, y = train[features], train[TARGET]
X_test = test[features]

# Пропуски: CatBoost сам справляется с NaN для числовых
# Категории: указать cat_features или сделать LabelEncoder

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = CatBoostClassifier(iterations=2000, learning_rate=0.05, depth=6,
                           eval_metric='F1', early_stopping_rounds=100, verbose=200)
model.fit(X_tr, y_tr, eval_set=(X_val, y_val))

print(classification_report(y_val, model.predict(X_val)))

sub = pd.DataFrame({ID_COL: test[ID_COL], TARGET: model.predict(X_test)})
sub.to_csv('submission.csv', index=False)

---
## 2. Картинки -> ResNet fine-tuning

In [ ]:
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image

IMG_SIZE = 224
train_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.RandomHorizontalFlip(),
                       T.ColorJitter(0.2, 0.2, 0.2), T.ToTensor(),
                       T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
val_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(),
                     T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

class ImgDS(Dataset):
    def __init__(self, paths, labels=None, tf=None):
        self.paths, self.labels, self.tf = paths, labels, tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = self.tf(Image.open(self.paths[i]).convert('RGB'))
        return (img, self.labels[i]) if self.labels is not None else img

NUM_CLASSES = 10  # <--
model = models.resnet18(weights='IMAGENET1K_V1')
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

# train_loader = DataLoader(ImgDS(train_paths, train_labels, train_tf), batch_size=32, shuffle=True)
# val_loader = DataLoader(ImgDS(val_paths, val_labels, val_tf), batch_size=64, shuffle=False)

In [ ]:
best_f1 = 0
for epoch in range(15):
    model.train()
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()

    model.eval()
    preds, gt = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            preds.extend(model(imgs.to(device)).argmax(1).cpu().numpy())
            gt.extend(labels.numpy())
    f1 = f1_score(gt, preds, average='macro')
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), 'best.pt')
    print(f'Epoch {epoch+1} val_f1={f1:.4f}')

model.load_state_dict(torch.load('best.pt', map_location=device))

---
## 3. Текст -> TF-IDF (быстро) или Transformers (точно)

In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

def clean(t):
    t = str(t).lower()
    t = re.sub(r'https?://\S+', '', t)
    t = re.sub(r'[^\w\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

# --- Быстрый бейзлайн ---
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True)),
    ('clf', LogisticRegression(C=1.0, max_iter=1000)),
])
# pipe.fit(train['text'].apply(clean), train[TARGET])
# preds = pipe.predict(test['text'].apply(clean))

In [ ]:
# --- Transformers (если есть GPU и время) ---
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup

MODEL_NAME = 'bert-base-uncased'  # или cointegrated/rubert-tiny2
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDS(Dataset):
    def __init__(self, texts, labels=None):
        self.texts, self.labels = texts, labels
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = tokenizer(self.texts[i], max_length=128, padding='max_length', truncation=True, return_tensors='pt')
        item = {k: v.squeeze() for k, v in enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[i], dtype=torch.long)
        return item

nlp_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_CLASSES, ignore_mismatched_sizes=True
).to(device)

# Дальше стандартный train loop -- см. nlp.ipynb

---
## 4. Кластеризация (если нет таргета)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, ts
from sklearn.metrics import silhouette_score

X_scaled = StandardScaler().fit_transform(X)

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)
print(f'Silhouette: {silhouette_score(X_scaled, labels):.4f}')

# Визуализация
import matplotlib.pyplot as plt
X_2d = PCA(n_components=2).fit_transform(X_scaled)
plt.scatter(X_2d[:,0], X_2d[:,1], c=labels, cmap='tab10', s=5, alpha=0.5)
plt.show()

---
## 5. Трюки (добавить к любому решению)

In [ ]:
# --- KFold (для CatBoost/XGBoost/sklearn) ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(X)); test_avg = np.zeros(len(X_test))

for fold, (ti, vi) in enumerate(skf.split(X, y)):
    m = CatBoostClassifier(iterations=2000, learning_rate=0.05, depth=6,
                           early_stopping_rounds=100, verbose=0)
    m.fit(X.iloc[ti], y.iloc[ti], eval_set=(X.iloc[vi], y.iloc[vi]))
    oof[vi] = m.predict(X.iloc[vi])
    test_avg += m.predict(X_test) / 5

print(f'OOF F1: {f1_score(y, oof, average="macro"):.4f}')

In [ ]:
# --- Ансамбль ---
# probs1, probs2 -- np.array [n, classes] от разных моделей
# final = (0.6 * probs1 + 0.4 * probs2).argmax(axis=1)

In [ ]:
# --- Несбалансированные классы ---
from sklearn.utils.class_weight import compute_class_weight
w = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32).to(device))

---
## 6. Submission

In [ ]:
sub = pd.DataFrame({ID_COL: test[ID_COL], TARGET: test_preds})
sub.to_csv('submission.csv', index=False)
sub.head()